# 04 — Deep Learning Models: LSTM, STGCN, DCRNN
Train and evaluate all deep models. LSTM (no graph), STGCN and DCRNN (with graph).

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np, torch
from src import config
from src.config import set_seed, get_device
from src.data_loader import prepare_dataset
from src.graph_builder import build_graph
from src.train import train_model, predict_model
from src.evaluate import evaluate_predictions, print_results, compare_models, save_results
set_seed(42); device = get_device()

Using GPU: NVIDIA GeForce RTX 4060
GPU Memory: 8.2 GB


In [2]:
data = prepare_dataset(config.METR_LA_PATH, batch_size=config.BATCH_SIZE)
mean, std = data['mean'], data['std']
num_sensors = data['splits']['train'][0].shape[2]
train_end = int(len(data['raw_data']) * config.TRAIN_RATIO)
graph = build_graph(data['raw_data'][:train_end], sigma=config.GRAPH_SIGMA, epsilon=config.GRAPH_EPSILON)
all_results, all_preds, histories = {}, {}, {}

Loading /home/anonymous/GraphNN/dataset/METR-LA.csv...
  Shape: (34272, 207) (timesteps × sensors)
  Time range: 2012-03-01 00:00:00 to 2012-06-27 23:55:00
  Missing values: 0 (0.00%)
  Zero values: 575302 (8.11%)
Handling missing values...
  After cleaning — NaN: 0, Zeros: 0
Creating sequences...
  Created 34249 sequences: X=(34249, 12, 207), Y=(34249, 12, 207)
Splitting data...
  train: 23974 samples
  val: 3425 samples
  test: 6850 samples
Building graph from sensor correlations...
  Adjacency matrix: 207x207
  Non-zero entries: 447 / 42849
  Sparsity: 98.96%
  Avg connections per node: 2.2
  Computing Chebyshev polynomials...
  Computing diffusion matrices...
Graph construction complete.



## 1. LSTM — No Graph, ~5 min

In [ ]:
from src.models.lstm_model import LSTMModel
lstm = LSTMModel(num_sensors, config.SEQ_LEN, config.PRED_LEN, config.LSTM_HIDDEN, config.LSTM_LAYERS, config.LSTM_DROPOUT)
print(f"LSTM params: {sum(p.numel() for p in lstm.parameters()):,}")
h = train_model(lstm, data['loaders']['train'], data['loaders']['val'], config, 'lstm', 'METR-LA')
p, gt = predict_model(lstm, data['loaders']['test'], config, 'lstm')
r = evaluate_predictions(p, gt, mean, std); print_results(r, 'LSTM')
all_results['LSTM'] = r; all_preds['LSTM'] = p; histories['LSTM'] = h

## 2. STGCN — ChebNet + Temporal Conv, ~5 min

In [ ]:
from src.models.stgcn import STGCN
stgcn = STGCN(num_sensors, config.SEQ_LEN, config.PRED_LEN, K=config.STGCN_K, channels=config.STGCN_CHANNELS)
print(f"STGCN params: {sum(p.numel() for p in stgcn.parameters()):,}")
h = train_model(stgcn, data['loaders']['train'], data['loaders']['val'], config, 'stgcn', 'METR-LA', graph_data=graph)
p, gt = predict_model(stgcn, data['loaders']['test'], config, 'stgcn', graph_data=graph)
r = evaluate_predictions(p, gt, mean, std); print_results(r, 'STGCN')
all_results['STGCN'] = r; all_preds['STGCN'] = p; histories['STGCN'] = h

## 3. DCRNN — Diffusion Conv + GRU Seq2Seq, ~10 min

In [ ]:
from src.models.dcrnn import DCRNN
n_sup = len(graph['diffusion_supports'])
dcrnn = DCRNN(num_sensors, n_sup, config.SEQ_LEN, config.PRED_LEN, config.DCRNN_HIDDEN, config.DCRNN_LAYERS)
print(f"DCRNN params: {sum(p.numel() for p in dcrnn.parameters()):,}")
h = train_model(dcrnn, data['loaders']['train'], data['loaders']['val'], config, 'dcrnn', 'METR-LA', graph_data=graph)
p, gt = predict_model(dcrnn, data['loaders']['test'], config, 'dcrnn', graph_data=graph)
r = evaluate_predictions(p, gt, mean, std); print_results(r, 'DCRNN')
all_results['DCRNN'] = r; all_preds['DCRNN'] = p; histories['DCRNN'] = h

## 4. Compare & Training Curves

In [ ]:
compare_models(all_results, 'METR-LA')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(histories), figsize=(6*len(histories), 4))
if len(histories)==1: axes=[axes]
colors = {'LSTM':'#2ecc71','STGCN':'#3498db','DCRNN':'#9b59b6'}
for i,(n,h) in enumerate(histories.items()):
    c=colors.get(n,'#333')
    axes[i].plot(h['train_loss'],label='Train',color=c)
    axes[i].plot(h['val_loss'],label='Val',color=c,linestyle='--')
    axes[i].set_title(f'{n} Loss'); axes[i].set_xlabel('Epoch'); axes[i].legend(); axes[i].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig('../results/plots/deep_training.png',dpi=150); plt.show()

In [ ]:
save_results(all_results, '../results/metrics', 'METR-LA_deep')
print("Deep model results saved!")